# 异构计算引言与概述

为什么训练和推理一个大模型，不能只靠 CPU？本节从神经网络计算的本质特点出发，解释 CPU 在高性能计算场景下的不足，进而引出异构计算的必要性，并建立 **Host + Device** 的基本分工模型。这也是你学习任何 NPU 编程框架的第一课。


## 1. 神经网络计算有什么特点

神经网络的前向与反向计算，绝大部分时间都花在**少数几类高度重复的数学运算**上：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">运算类型</th><th style="text-align:left;">典型场景</th><th style="text-align:left;">特点</th></tr>
<tr><td>矩阵乘（GEMM/批量 GEMM）</td><td>全连接层、注意力 QKV、大模型的核心耗时</td><td>规整、可并行、算力密集</td></tr>
<tr><td>逐元素运算</td><td>激活函数、残差相加、缩放</td><td>数据同构、操作一致</td></tr>
<tr><td>归约运算</td><td>Softmax、LayerNorm、损失统计</td><td>沿某一维度合并结果</td></tr>
</table>

这些运算有两个共同点：**数据量大**（现代大模型参数量从数十亿到万亿级，一次前向要处理成千上万个 Token），以及**对每个数据元素执行的操作高度一致**。例如一次矩阵乘中，同一个"乘加"动作会被重复执行数万亿次。

> 换个角度：神经网络计算 ≈ 海量数据 × 少数几种重复操作。这正是一种**吞吐型（Throughput）**负载——我们更关心"单位时间能处理多少数据"，而不是"单个任务多快完成"。


## 2. CPU 的取舍：为通用与低延迟而生

CPU 是通用处理器，它的设计目标是在**串行、不规则、分支密集**的负载上取得好的表现。为此，CPU 把大量晶体管花在了：

- **复杂的控制逻辑**：分支预测、乱序执行、指令调度，用于加速不规则的串行程序；
- **大容量多级缓存**：用于缓解不规则访存的延迟；
- **少量强核心**：CPU 核心强调通用计算能力，不能仅按核心数与加速器的计算单元比较吞吐量。

这套设计面向的是**延迟敏感（Latency）**的通用计算。但面对"海量数据 × 重复操作"的吞吐型负载，CPU 的问题就暴露出来：

1. **并行度不足**：即使利用多核与向量指令，大规模矩阵计算的吞吐仍受计算单元配置、带宽和功耗预算限制；
2. **资源分配不同**：控制逻辑对通用程序有价值，但在规整矩阵运算中的收益与专用矩阵计算单元不同；
3. **功耗效率低**：对适合加速器的大规模规整负载，专用计算单元通常具有更高能效；小规模或控制密集任务仍需具体分析。

这并不是 CPU 的缺陷，而是**取舍**：它被设计成"什么都能干"，而不是"把某类事情做到极致"。


## 3. 异构计算：让专门的芯片干专门的事

**异构计算（Heterogeneous Computing）** 的思路是：系统中存在多种处理器，各司其职——

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">角色</th><th style="text-align:left;">处理器</th><th style="text-align:left;">擅长</th><th style="text-align:left;">在本课程语境下</th></tr>
<tr><td><b>Host（主机侧）</b></td><td>CPU</td><td>控制流、任务编排、数据准备、框架调度</td><td>准备输入 Tensor、启动 Kernel、读取结果</td></tr>
<tr><td><b>Device（设备侧）</b></td><td>NPU（昇腾）</td><td>大规模规整并行计算</td><td>执行 Kernel，完成矩阵/矢量计算</td></tr>
</table>

一次典型的异构计算流程如下：

1. Host 侧程序（如 PyTorch 脚本）在 CPU 上准备输入数据，并把数据放到 Device 可访问的内存中；
2. Host 侧向 Device **下发计算任务**（本课程中即启动一个 PyPTO Pro Kernel）；
3. Device 上的大量计算核心并行执行任务；
4. Host 侧等待任务完成后，读取并验证结果。

这套分工你其实早已在使用：`x = torch.randn(..., device="npu:0")` 中，`randn` 的调度发生在 CPU，而生成的张量数据与后续算子计算发生在 NPU 上——**PyTorch + torch_npu 的日常使用，本身就是异构计算**。本课程做的事，是让你从"使用现成算子"进一步走到"自己编写 Device 侧的计算程序"，把神经网络中的某段计算（某个算子）亲手写到 NPU 上。


## 4. 算子基本概念：全课程的通用词汇

后续章节将频繁使用以下术语，它们与 Ascend C 等算子开发体系通用（本课程语境下的含义）：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">术语</th><th style="text-align:left;">含义</th><th style="text-align:left;">本课程中的对应</th></tr>
<tr><td><b>算子（Operator）</b></td><td>对 Tensor 执行一次特定计算的函数单元，是 NPU 计算的基本交付物</td><td>一个 Pro Kernel 即一个算子的 Device 侧实现</td></tr>
<tr><td><b>算子名称 / 类型</b></td><td>Name 是网络中的唯一标识；Type 决定计算逻辑（Add、Matmul、Softmax…）</td><td>Kernel 函数名（name 参数可自定义）</td></tr>
<tr><td><b>Tensor（张量）</b></td><td>多维数据的容器；本课程中特指 GM 中的数据视图（第 2 章）</td><td><code>pl.Tensor[[shape], dtype]</code></td></tr>
<tr><td><b>Shape（形状）</b></td><td>各维度长度，如 [8, 256]</td><td>Tensor 标注与 TileType 的 shape 参数</td></tr>
<tr><td><b>dtype（数据类型）</b></td><td>元素的数据类型（FP16/BF16/FP32/INT8…）</td><td><code>pl.DT_FP32</code> 等</td></tr>
<tr><td><b>Format / Layout（排布）</b></td><td>数据在内存中的物理排布方式（ND 线性、NZ/ZN 分形等）</td><td>Tile 的 layout、分形（第 4 章）</td></tr>
<tr><td><b>Axis（轴）</b></td><td>张量的维度编号（0 起算）；归约方向沿轴进行</td><td>Tensor 的 <code>shape[axis]</code>；Tile 归约的 <code>dim</code> 是专门的方向编码，不能直接套用 NumPy/PyTorch 轴号（见 03.03）</td></tr>
</table>

> 记忆锚点：**算子 = 有名字、有类型、吃 Tensor、吐 Tensor 的计算函数**；shape/dtype/format/axis 是描述 Tensor 的四件套。


### 算例：同样是加法，为什么还要考虑数据搬运？

处理两个 [8,256] FP32 输入时，共有 2048 次元素加法。每个输入占 8192 B，输出也占 8192 B；仅计算一次理想 GM 读写量就有 24576 B。Host 还要准备数据、下发任务并等待结果，所以“小张量也必然在 NPU 上更快”并不能由并行度推出。

把问题拆成三份规格：数学上每个输出只依赖同位置两个输入；存储上输入输出各是一段连续数组；执行上 Host 负责启动，Device 负责全部 2048 个元素。后面若把输出改成每行一个和，数学依赖与输出形状都变了，不能只替换函数名。

**自查**：如果输入改成 FP16，元素加法次数仍为 2048，理想 GM 字节数变成 12288 B。这个计算只描述流量，不给出实际延迟。


## 5. 课后练习

1. （多选）以下哪些特点属于神经网络计算的典型负载特征？
   A. 分支密集且不可预测<br/>
   B. 数据量大<br/>
   C. 对每个元素执行的操作高度一致<br/>
   D. 以串行依赖为主
2. （填空）异构计算中，负责准备数据、下发任务的角色称为 ______（Host/Device 二选一），负责大规模并行计算的角色称为 ______。
3. （选择）CPU 把大量晶体管用于分支预测、乱序执行与大缓存，主要服务于哪类负载？
   A. 吞吐型规整计算<br/>
   B. 串行、不规则的通用计算
4. （填空）描述 Tensor 的四个基本属性是 ______（形状）、______（数据类型）、______（排布）和 ______（轴）。

In [ ]:
!cat ./answer/01.02_answer.txt

## 6. 本节小结

- 神经网络计算 ≈ **海量数据 × 少数重复操作**，是吞吐型负载；
- CPU 为通用与低延迟而生，面对适合加速器的规整大规模负载时，专用芯片通常更有吞吐与能效优势；
- **异构计算 = Host（CPU，编排）+ Device（NPU，计算）**；你每天都在用 PyTorch 体验它，本课程带你成为 Device 侧程序的编写者。

下一节，我们打开 Device 的"引擎盖"，看看 Ascend 950 的 AI Core 是如何组织的。